In [1]:
import os
import sys
import torch
import numpy as np

TOPICS TO COVER:

- Off-policy learning
    - Importance Sampling
    - Maximum Entropy Reinforcement Learning
    - Imitation Learning

- On-policy learning
    - Model free RL
        - Value based methods
            - [CODE] Monte Carlo RL
            - [CODE] Temporal Difference RL (SARSA)
            - [CODE] Q learning
        - Policy search methods
            - Policy Gradient theorem
            - [CODE] REINFORCE
            - [CODE] A2C
            - [CODE] PPO
            - [CODE] GRPO 
            - [CODE] TRPO
    - Model-based RL
        - [CODE] MPC
        - World Models

> **Category:** Off-Policy | Value-Based | Model-Free
> 
> **The Concept:** **Learn by waiting until the end.** We play a whole game, look at the final score, and then go back and tell every step we took, "Hey, this led to a win!" or "This led to a loss!"
> 
> **The Steps:**
> 1. **Initialize:** Create a table (Q-Table) to store a "score" for every possible move.
> 2. **Play:** Start a game and play it until it is 100% over (win, lose, or time out). 
> 3. **Record:** Keep a list of every state you saw, every action you took, and every reward you got.
> 4. **Calculate:** When the game ends, calculate the total "Score" ($G$).
> 5. **Update:** For every move you made in that game:
>    - Add the final score to a running average for that specific move.
>    - $Q(S_t, A_t) \leftarrow \text{average}(Returns)$
> 6. **Repeat:** Do this for thousands of games until the table points toward the highest rewards.

In [13]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import numpy as np
import random
import os
from IPython.display import Video, display

# --- 1. INITIALIZATION ---
def train_monte_carlo():
    # Setup standard environment for training
    env = gym.make("FrozenLake-v1", is_slippery=False)
    state_size = env.observation_space.n
    action_size = env.action_space.n
    
    q_table = np.zeros((state_size, action_size))
    returns_count = np.zeros((state_size, action_size))
    
    gamma = 0.95
    epsilon = 1.0
    epsilon_decay = 0.99
    min_epsilon = 0.1
    episodes = 2000

    # --- 2. THE TRAINING LOOP ---
    print("Training started...")
    for episode in range(episodes):
        state, _ = env.reset()
        episode_data = [] 
        done = False
        
        while not done:
            if random.uniform(0, 1) < epsilon:
                action = env.action_space.sample()
            else:
                action = np.argmax(q_table[state])
            
            next_state, reward, term, trunc, _ = env.step(action)
            episode_data.append((state, action, reward))
            state = next_state
            done = term or trunc
            
        G = 0
        for i in range(len(episode_data)-1, -1, -1):
            s, a, r = episode_data[i]
            G = r + gamma * G 
            returns_count[s, a] += 1
            q_table[s, a] += (1 / returns_count[s, a]) * (G - q_table[s, a])
            
        epsilon = max(min_epsilon, epsilon * epsilon_decay)

        if (episode + 1) % 500 == 0:
            print(f"Episode {episode + 1} | Last Return: {G:.2f} | Epsilon: {epsilon:.2f}")

    env.close()
    print("Training Complete!\n")

    # --- 3. VISUALIZATION & EMBEDDING ---
    print("Recording final result...")
    video_path = "./mc_videos"
    video_env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="rgb_array")
    
    # Wrap it to record
    video_env = RecordVideo(
        video_env, 
        video_folder=video_path, 
        name_prefix="mc_result",
        episode_trigger=lambda x: True 
    )

    state, _ = video_env.reset()
    done = False
    while not done:
        action = np.argmax(q_table[state]) 
        state, reward, term, trunc, _ = video_env.step(action)
        done = term or trunc
    
    video_env.close()

    # --- 4. RENDER VIDEO IN JUPYTER ---
    # Find the mp4 file in the folder
    mp4_files = [f for f in os.listdir(video_path) if f.endswith(".mp4")]
    if mp4_files:
        # Get the latest one
        latest_video = os.path.join(video_path, sorted(mp4_files)[-1])
        print(f"\nPlaying Video: {latest_video}")
        display(Video(latest_video, embed=True, width=400))
    else:
        print("Video file not found.")
    
    return q_table

# Run the process
final_q_table = train_monte_carlo()

Training started...
Episode 500 | Last Return: 0.77 | Epsilon: 0.10
Episode 1000 | Last Return: 0.77 | Epsilon: 0.10
Episode 1500 | Last Return: 0.00 | Epsilon: 0.10
Episode 2000 | Last Return: 0.74 | Epsilon: 0.10
Training Complete!

Recording final result...

Playing Video: ./mc_videos/mc_result-episode-0.mp4


> **Category:** On-Policy | Value-Based | Model-Free
> 
> **The Concept:** **Learn step-by-step.** Instead of waiting for the end of the game, we update our guess after every single move based on the immediate reward and our "guess" for the next move.
> 
> **The Steps:**
> 1. **Start:** Be in a state ($S$) and pick an action ($A$).
> 2. **Step:** Do the action. See the reward ($R$) and the new state ($S'$).
> 3. **Look Ahead:** Pick the **next** action ($A'$) you are actually going to do based on your current table.
> 4. **Update:** Adjust your current move's value using this logic:
>    - *New Guess* = Reward + (Current estimate of the value of $S', A'$).
>    - *Update*: $Q(S, A) \leftarrow Q(S, A) + \alpha [R + \gamma Q(S', A') - Q(S, A)]$
> 5. **Move On:** Your "Next Action" ($A'$) becomes your "Current Action" ($A$). Repeat for every step.

In [14]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import numpy as np
import random
import os
from IPython.display import Video, display

# --- 1. INITIALIZATION ---
def train_sarsa_td():
    # Setup training environment (no rendering for speed)
    env = gym.make("FrozenLake-v1", is_slippery=False)
    state_size = env.observation_space.n
    action_size = env.action_space.n
    
    # Initialize Q-table
    q_table = np.zeros((state_size, action_size))

    # Hyperparameters
    alpha = 0.1        # Learning rate
    gamma = 0.95       # Discount factor
    epsilon = 1.0      # Exploration
    epsilon_decay = 0.99
    episodes = 1000

    # --- 2. TRAINING LOOP ---
    print("Training SARSA agent...")
    for episode in range(episodes):
        state, _ = env.reset()
        
        # SARSA: Choose FIRST action before the loop
        if random.uniform(0, 1) < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table[state])
            
        done = False
        while not done:
            next_state, reward, term, trunc, _ = env.step(action)
            done = term or trunc
            
            # SARSA: Choose NEXT action (On-Policy)
            if random.uniform(0, 1) < epsilon:
                next_action = env.action_space.sample()
            else:
                next_action = np.argmax(q_table[next_state])
            
            # TD Update: Use value of the action we actually picked (next_action)
            td_target = reward + gamma * q_table[next_state, next_action]
            td_error = td_target - q_table[state, action]
            q_table[state, action] += alpha * td_error
            
            # Move to the next state and action
            state = next_state
            action = next_action
            
        epsilon = max(0.01, epsilon * epsilon_decay)

        if (episode + 1) % 250 == 0:
            print(f"Episode {episode+1} | Epsilon: {epsilon:.2f} | Table Mean: {np.mean(q_table):.4f}")

    env.close()
    print("Training Complete!\n")

    # --- 3. VISUALIZATION & EMBEDDING ---
    print("Recording SARSA final result...")
    video_folder = "./sarsa_videos"
    # Create new env with 'rgb_array' for video capture
    video_env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="rgb_array")
    
    # Wrap to record video
    video_env = RecordVideo(
        video_env, 
        video_folder=video_folder, 
        name_prefix="sarsa_result",
        episode_trigger=lambda x: True 
    )

    # Test the learned policy
    state, _ = video_env.reset()
    done = False
    while not done:
        action = np.argmax(q_table[state]) # Follow best learned moves
        state, reward, term, trunc, _ = video_env.step(action)
        done = term or trunc
    
    video_env.close()

    # --- 4. RENDER VIDEO IN JUPYTER ---
    mp4_files = [f for f in os.listdir(video_folder) if f.endswith(".mp4")]
    if mp4_files:
        # Sort files to get the most recent recording
        latest_video = os.path.join(video_folder, sorted(mp4_files)[-1])
        print(f"\nPlaying Video: {latest_video}")
        display(Video(latest_video, embed=True, width=400))
    else:
        print("Video file not found.")

    return q_table

# Run it
final_q_table = train_sarsa_td()

Training SARSA agent...
Episode 250 | Epsilon: 0.08 | Table Mean: 0.0000
Episode 500 | Epsilon: 0.01 | Table Mean: 0.0000
Episode 750 | Epsilon: 0.01 | Table Mean: 0.0000
Episode 1000 | Epsilon: 0.01 | Table Mean: 0.0000
Training Complete!

Recording SARSA final result...

Playing Video: ./sarsa_videos/sarsa_result-episode-0.mp4


> **Category:** Off-Policy | Value-Based | Model-Free
> 
> **The Concept:** **Learn the "Perfect" path.** Similar to SARSA, but instead of updating based on what we *actually* do next, we update based on the *best possible* thing we could do next.
> 
> **The Steps:**
> 1. **Start:** Be in a state ($S$).
> 2. **Step:** Pick an action ($A$), do it, and see the reward ($R$) and new state ($S'$).
> 3. **The "Best" Logic:** Look at the new state ($S'$) and find the action with the highest score in the table.
> 4. **Update:** 
>    - *Target* = Reward + (Score of the absolute best move possible in $S'$).
>    - *Update*: $Q(S, A) \leftarrow Q(S, A) + \alpha [R + \gamma \max_{a} Q(S', a) - Q(S, A)]$
> 5. **Repeat:** This allows the agent to learn the "optimal" strategy while it is still running around randomly exploring.

In [15]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import numpy as np
import random
import os
from IPython.display import Video, display

# --- 1. INITIALIZATION ---
def train_q_learning():
    # Setup standard environment for training (no rendering for speed)
    env = gym.make("FrozenLake-v1", is_slippery=False)
    
    state_size = env.observation_space.n
    action_size = env.action_space.n
    q_table = np.zeros((state_size, action_size))

    # Hyperparameters
    learning_rate = 0.1    
    gamma = 0.95           
    epsilon = 1.0          
    epsilon_decay = 0.995  
    min_epsilon = 0.01
    episodes = 1000

    # --- 2. TRAINING LOOP ---
    print("Training Q-Learning agent...")
    for episode in range(episodes):
        state, _ = env.reset()
        done = False
        total_reward = 0
        
        while not done:
            if random.uniform(0, 1) < epsilon:
                action = env.action_space.sample() 
            else:
                action = np.argmax(q_table[state]) 
            
            next_state, reward, term, trunc, _ = env.step(action)
            done = term or trunc
            
            # Best possible Q-value for the next state
            best_next_q = np.max(q_table[next_state])
            
            # Update the specific (state, action) pair
            td_target = reward + gamma * best_next_q
            td_error = td_target - q_table[state, action]
            q_table[state, action] += learning_rate * td_error
            
            state = next_state
            total_reward += reward
            
        epsilon = max(min_epsilon, epsilon * epsilon_decay)
        
        if (episode + 1) % 250 == 0:
            print(f"Episode {episode + 1} | Epsilon: {epsilon:.2f} | Last Reward: {total_reward}")

    env.close()
    print("Training Complete!\n")

    # --- 3. VISUALIZATION & EMBEDDING ---
    print("Recording final Q-Learning result...")
    video_folder = "./q_learning_videos"
    # Create new env with 'rgb_array' for video capture
    video_env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="rgb_array")
    
    # Wrap to record video
    video_env = RecordVideo(
        video_env, 
        video_folder=video_folder, 
        name_prefix="q_learning_result",
        episode_trigger=lambda x: True 
    )

    # Test the learned policy
    state, _ = video_env.reset()
    done = False
    while not done:
        action = np.argmax(q_table[state]) # Follow the optimal policy
        state, reward, term, trunc, _ = video_env.step(action)
        done = term or trunc
    
    video_env.close()

    # --- 4. RENDER VIDEO IN JUPYTER ---
    mp4_files = [f for f in os.listdir(video_folder) if f.endswith(".mp4")]
    if mp4_files:
        # Sort files to ensure we get the latest recording
        latest_video = os.path.join(video_folder, sorted(mp4_files)[-1])
        print(f"\nPlaying Video: {latest_video}")
        display(Video(latest_video, embed=True, width=400))
    else:
        print("Video file not found.")
    
    return q_table

# Run the full process
final_q_table = train_q_learning()

Training Q-Learning agent...
Episode 250 | Epsilon: 0.29 | Last Reward: 0
Episode 500 | Epsilon: 0.08 | Last Reward: 0
Episode 750 | Epsilon: 0.02 | Last Reward: 0
Episode 1000 | Epsilon: 0.01 | Last Reward: 0
Training Complete!

Recording final Q-Learning result...

Playing Video: ./q_learning_videos/q_learning_result-episode-0.mp4


> **Category:** On-Policy | Policy Search | Model-Free
> 
> **The Concept:** **If it worked, do it more.** We don't use a table of scores. Instead, we have a neural network that outputs probabilities. If the game ends well, we tweak the network to make those specific moves more likely.
> 
> **The Steps:**
> 1. **Play:** Run an entire episode using your network to pick moves.
> 2. **Store:** Save the "Probability" $(\pi)$ of every move and the rewards you got.
> 3. **Wait:** Wait until the game is over to see the total score ($G$).
> 4. **Reward/Punish:**
>    - If score is high: Tweak the network to make every move in that game *more* likely.
>    - If score is low: Tweak it to make those moves *less* likely.
> 5. **Update:** $\theta \leftarrow \theta + \alpha G \nabla_{\theta} \log \pi_{\theta}(A_t | S_t)$

In [16]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
from IPython.display import Video, display

# --- 1. THE POLICY NETWORK ---
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(PolicyNetwork, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim),
            nn.Softmax(dim=-1) 
        )

    def forward(self, x):
        return self.net(x)

# --- 2. THE TRAINING LOOP ---
def train_reinforce():
    env = gym.make("CartPole-v1")
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    policy = PolicyNetwork(state_dim, action_dim)
    optimizer = optim.Adam(policy.parameters(), lr=0.01)
    gamma = 0.99

    print("Training REINFORCE agent...")
    for episode in range(500):
        state, _ = env.reset()
        log_probs = []
        rewards = []
        done = False
        
        while not done:
            state_t = torch.FloatTensor(state).unsqueeze(0)
            probs = policy(state_t)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            
            next_state, reward, term, trunc, _ = env.step(action.item())
            done = term or trunc
            
            log_probs.append(dist.log_prob(action))
            rewards.append(reward)
            state = next_state
        
        # Calculate Returns
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        
        returns = torch.FloatTensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-6)

        # Policy Gradient Update
        loss = []
        for log_prob, Gt in zip(log_probs, returns):
            loss.append(-log_prob * Gt)
        
        optimizer.zero_grad()
        policy_loss = torch.cat(loss).sum()
        policy_loss.backward()
        optimizer.step()
        
        if episode % 100 == 0:
            print(f"Episode {episode} | Total Reward: {sum(rewards)}")

    env.close()
    print("Training Complete!\n")

    # --- 3. VISUALIZATION & EMBEDDING ---
    print("Recording REINFORCE final result...")
    video_folder = "./reinforce_videos"
    video_env = gym.make("CartPole-v1", render_mode="rgb_array")
    
    # Wrap to record video
    video_env = RecordVideo(
        video_env, 
        video_folder=video_folder, 
        name_prefix="reinforce_result",
        episode_trigger=lambda x: True
    )

    policy.eval() # Set to evaluation mode
    state, _ = video_env.reset()
    done = False
    total_eval_reward = 0
    
    while not done:
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0)
            probs = policy(state_t)
            # Pick the action with highest probability for the video
            action = torch.argmax(probs).item()
            
        state, reward, term, trunc, _ = video_env.step(action)
        total_eval_reward += reward
        done = term or trunc
    
    video_env.close()

    # --- 4. RENDER VIDEO IN JUPYTER ---
    mp4_files = [f for f in os.listdir(video_folder) if f.endswith(".mp4")]
    if mp4_files:
        latest_video = os.path.join(video_folder, sorted(mp4_files)[-1])
        print(f"\nFinal Reward in Video: {total_eval_reward}")
        print(f"Playing Video: {latest_video}")
        display(Video(latest_video, embed=True, width=600))
    else:
        print("Video file not found.")

    return policy

# Run the full process
trained_policy = train_reinforce()

Training REINFORCE agent...
Episode 0 | Total Reward: 15.0
Episode 100 | Total Reward: 112.0
Episode 200 | Total Reward: 500.0
Episode 300 | Total Reward: 500.0
Episode 400 | Total Reward: 500.0
Training Complete!

Recording REINFORCE final result...

Final Reward in Video: 500.0
Playing Video: ./reinforce_videos/reinforce_result-episode-0.mp4


> **Category:** On-Policy | Policy Search | Model-Free
> 
> **The Concept:** **The "Coach" and the "Athlete."** The **Actor** (Athlete) tries to play, and the **Critic** (Coach) watches and says "That move was better than average!"
> 
> **The Steps:**
> 1. **Move:** Actor picks an action. Environment gives a reward and a new state.
> 2. **Evaluate:** Critic looks at the state and gives its guess for the expected score ($V$).
> 3. **Calculate "Advantage":**
>    - $Advantage = (R + \gamma V(S')) - V(S)$
>    - If $Advantage$ is positive, the move was a pleasant surprise!
> 4. **Learn:**
>    - **Actor**: Makes that move more likely because the Advantage was positive.
>    - **Critic**: Updates its "Guessing" skills to be more accurate next time.

In [17]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
from IPython.display import Video, display

# --- 1. THE ACTOR-CRITIC NETWORK ---
class A2CNet(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(A2CNet, self).__init__()
        self.common = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU()
        )
        # Actor: Outputs probability distribution over actions
        self.actor = nn.Linear(128, action_dim)
        # Critic: Outputs a single value V(s) estimating expected return
        self.critic = nn.Linear(128, 1)

    def forward(self, x):
        x = self.common(x)
        probs = torch.softmax(self.actor(x), dim=-1)
        value = self.critic(x)
        return probs, value

# --- 2. THE TRAINING LOOP ---
def train_a2c():
    env = gym.make("CartPole-v1")
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    model = A2CNet(state_dim, action_dim)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    gamma = 0.99 

    print("Training A2C agent...")
    for episode in range(501):
        state, _ = env.reset()
        log_probs = []
        values = []
        rewards = []
        done = False
        
        # --- DATA COLLECTION ---
        while not done:
            state_t = torch.FloatTensor(state).unsqueeze(0)
            probs, value = model(state_t)
            
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            
            next_state, reward, term, trunc, _ = env.step(action.item())
            done = term or trunc
            
            log_probs.append(dist.log_prob(action))
            values.append(value)
            rewards.append(reward)
            
            state = next_state
        
        # --- CALCULATION ---
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        
        returns = torch.FloatTensor(returns).detach()
        values = torch.cat(values).squeeze()
        log_probs = torch.stack(log_probs)
        
        # Advantage: How much better was the action than the average (Critic's guess)?
        advantage = returns - values

        # --- LOSS FUNCTIONS ---
        actor_loss = -(log_probs * advantage.detach()).mean()
        critic_loss = torch.nn.functional.mse_loss(values, returns)
        total_loss = actor_loss + 0.5 * critic_loss
        
        # --- UPDATE ---
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        if episode % 100 == 0:
            print(f"Episode {episode} | Total Reward: {sum(rewards)}")

    env.close()
    print("Training Complete!\n")

    # --- 3. VISUALIZATION & EMBEDDING ---
    print("Recording A2C final result...")
    video_folder = "./a2c_videos"
    video_env = gym.make("CartPole-v1", render_mode="rgb_array")
    
    # Wrap to record video
    video_env = RecordVideo(
        video_env, 
        video_folder=video_folder, 
        name_prefix="a2c_result",
        episode_trigger=lambda x: True
    )

    model.eval()
    state, _ = video_env.reset()
    done = False
    total_eval_reward = 0
    
    while not done:
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0)
            probs, _ = model(state_t)
            # Pick the best move for the demo
            action = torch.argmax(probs).item()
            
        state, reward, term, trunc, _ = video_env.step(action)
        total_eval_reward += reward
        done = term or trunc
    
    video_env.close()

    # --- 4. RENDER VIDEO IN JUPYTER ---
    mp4_files = [f for f in os.listdir(video_folder) if f.endswith(".mp4")]
    if mp4_files:
        latest_video = os.path.join(video_folder, sorted(mp4_files)[-1])
        print(f"\nFinal Reward in Video: {total_eval_reward}")
        print(f"Playing Video: {latest_video}")
        display(Video(latest_video, embed=True, width=600))
    else:
        print("Video file not found.")

    return model

# Run the full process
trained_a2c_model = train_a2c()

Training A2C agent...
Episode 0 | Total Reward: 20.0
Episode 100 | Total Reward: 15.0
Episode 200 | Total Reward: 41.0
Episode 300 | Total Reward: 59.0
Episode 400 | Total Reward: 65.0
Episode 500 | Total Reward: 141.0
Training Complete!

Recording A2C final result...

Final Reward in Video: 63.0
Playing Video: ./a2c_videos/a2c_result-episode-0.mp4


> **Category:** On-Policy | Policy Search | Model-Free
> 
> **The Concept:** **Don't change too fast.** PPO puts a "safety limit" on how much the brain can change in one update to keep training stable.
> 
> **The Steps:**
> 1. **Collect:** Run the agent for a while and save all the data (trajectories).
> 2. **Compare:** Look at the "New Brain" vs. the "Old Brain" policy ratio ($r_t$).
> 3. **The Clip:** If the change is too large, "Clip" the update so it stays within a small range (e.g., 20%).
> 4. **Update:** Optimize the clipped objective: $\mathbb{E}_t [\min(r_t(\theta)\hat{A}_t, \text{clamp}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat{A}_t)]$

In [18]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
from IPython.display import Video, display

# --- 1. THE NETWORK ARCHITECTURE ---
class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(ActorCritic, self).__init__()
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh()
        )
        self.actor = nn.Sequential(
            nn.Linear(64, action_dim),
            nn.Softmax(dim=-1)
        )
        self.critic = nn.Linear(64, 1)

    def forward(self, x):
        x = self.backbone(x)
        return self.actor(x), self.critic(x)

# --- 2. YOUR PPO UPDATE FUNCTION ---
def ppo_update(model, optimizer, states, actions, old_log_probs, targets, advantages):
    epsilon = 0.2
    for _ in range(10):
        probs, values = model(states)
        dist = torch.distributions.Categorical(probs)
        new_log_probs = dist.log_prob(actions)
        
        ratio = torch.exp(new_log_probs - old_log_probs)
        
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
        
        actor_loss = -torch.min(surr1, surr2).mean()
        critic_loss = torch.nn.functional.mse_loss(values.squeeze(), targets)
        
        optimizer.zero_grad()
        (actor_loss + critic_loss).backward()
        optimizer.step()

# --- 3. THE TRAINING LOOP ---
def train_ppo():
    env = gym.make("CartPole-v1")
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    model = ActorCritic(state_dim, action_dim)
    optimizer = optim.Adam(model.parameters(), lr=0.002)
    gamma = 0.99
    
    print("Training PPO agent...")
    for episode in range(200):
        states, actions, rewards, log_probs, values, masks = [], [], [], [], [], []
        state, _ = env.reset()
        episode_reward = 0
        
        for _ in range(200):
            state_t = torch.FloatTensor(state).unsqueeze(0)
            probs, val = model(state_t)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            
            next_state, reward, term, trunc, _ = env.step(action.item())
            done = term or trunc
            
            states.append(state_t)
            actions.append(action)
            rewards.append(reward)
            log_probs.append(dist.log_prob(action).detach())
            values.append(val.detach())
            masks.append(1 - done)
            
            state = next_state
            episode_reward += reward
            if done: break
            
        states = torch.cat(states)
        actions = torch.cat(actions)
        log_probs = torch.cat(log_probs)
        values = torch.cat(values).squeeze()
        
        returns = []
        G = 0
        for r, m in reversed(list(zip(rewards, masks))):
            G = r + gamma * G * m
            returns.insert(0, G)
            
        returns = torch.FloatTensor(returns)
        advantages = returns - values

        ppo_update(model, optimizer, states, actions, log_probs, returns, advantages)
        
        if episode % 40 == 0:
            print(f"Episode {episode} | Reward: {episode_reward}")

    env.close()
    print("Training Complete!\n")

    # --- 4. VISUALIZATION & EMBEDDING ---
    print("Recording PPO final result...")
    video_folder = "./ppo_v2_videos"
    # Create new env with 'rgb_array' for video capture
    video_env = gym.make("CartPole-v1", render_mode="rgb_array")
    
    video_env = RecordVideo(
        video_env, 
        video_folder=video_folder, 
        name_prefix="ppo_v2_result",
        episode_trigger=lambda x: True 
    )

    model.eval()
    state, _ = video_env.reset()
    done = False
    total_eval_reward = 0
    
    while not done:
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0)
            probs, _ = model(state_t)
            action = torch.argmax(probs).item() # Deterministic best move
            
        state, reward, term, trunc, _ = video_env.step(action)
        total_eval_reward += reward
        done = term or trunc
    
    video_env.close()

    # --- 5. RENDER VIDEO IN JUPYTER ---
    mp4_files = [f for f in os.listdir(video_folder) if f.endswith(".mp4")]
    if mp4_files:
        latest_video = os.path.join(video_folder, sorted(mp4_files)[-1])
        print(f"\nFinal Reward in Video: {total_eval_reward}")
        display(Video(latest_video, embed=True, width=600))
    else:
        print("Video file not found.")

    return model

# Start
trained_model = train_ppo()

Training PPO agent...
Episode 0 | Reward: 18.0
Episode 40 | Reward: 25.0
Episode 80 | Reward: 70.0
Episode 120 | Reward: 59.0
Episode 160 | Reward: 70.0
Training Complete!

Recording PPO final result...

Final Reward in Video: 109.0


> **Category:** On-Policy | Policy Search | Model-Free
> 
> **The Concept:** **Peer pressure.** Instead of a "Coach" (Critic), we try the same thing several times and compare them. If one attempt was better than the average of the group, we copy it.
> 
> **The Steps:**
> 1. **Group Up:** For one prompt, generate a **group** of $G$ different attempts.
> 2. **Score:** See the rewards/scores for all attempts in the group.
> 3. **Average:** Calculate the average score of that specific group.
> 4. **Relative Advantage:**
>    - $Advantage = (\text{Individual Reward} - \text{Group Mean}) / \text{Group StdDev}$
> 5. **Update:** Change the brain to favor the "above average" attempts.

In [19]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
from IPython.display import Video, display

# --- 1. THE POLICY NETWORK ---
class GRPO_Policy(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(GRPO_Policy, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        return self.net(x)

# --- 2. THE GRPO UPDATE LOGIC ---
def grpo_update(model, optimizer, states, actions, old_log_probs, rewards_group):
    # Normalize Rewards within the group (The "Relative" part of GRPO)
    mean_reward = np.mean(rewards_group)
    std_reward = np.std(rewards_group) + 1e-8
    advantages = torch.FloatTensor([(r - mean_reward) / std_reward for r in rewards_group])

    epsilon = 0.2
    probs = model(states)
    dist = torch.distributions.Categorical(probs)
    new_log_probs = dist.log_prob(actions)

    # Clipping logic
    ratio = torch.exp(new_log_probs - old_log_probs)
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages

    loss = -torch.min(surr1, surr2).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# --- 3. TRAINING LOOP ---
def train_grpo():
    env = gym.make("CartPole-v1")
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    model = GRPO_Policy(state_dim, action_dim)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    group_size = 16 # Group size for relative advantage
    
    print("Training GRPO agent...")
    for iteration in range(200):
        state, _ = env.reset()
        done = False
        total_iter_reward = 0
        
        # For simplicity, we sample a group per step to demonstrate the relative update
        while not done:
            state_t = torch.FloatTensor(state).repeat(group_size, 1)
            
            with torch.no_grad():
                probs = model(state_t)
                dist = torch.distributions.Categorical(probs)
                actions = dist.sample()
                old_log_probs = dist.log_prob(actions)
            
            # Take the first action in the group to progress the environment
            next_state, reward, term, trunc, _ = env.step(actions[0].item())
            done = term or trunc
            
            # Rewards for the group: 
            # In this demo, we mock variances in the group rewards 
            # (In LLMs, these would be different completions)
            rewards = [reward if i == 0 else (reward * np.random.uniform(0.8, 1.2)) for i in range(group_size)]
            
            # Update
            grpo_update(model, optimizer, state_t, actions, old_log_probs, rewards)
            
            state = next_state
            total_iter_reward += reward

        if iteration % 50 == 0:
            print(f"Iteration {iteration} | Last Episode Reward: {total_iter_reward}")

    env.close()
    print("Training Complete!\n")

    # --- 4. VISUALIZATION & EMBEDDING ---
    print("Recording GRPO final result...")
    video_folder = "./grpo_videos"
    video_env = gym.make("CartPole-v1", render_mode="rgb_array")
    
    video_env = RecordVideo(
        video_env, 
        video_folder=video_folder, 
        name_prefix="grpo_result",
        episode_trigger=lambda x: True
    )

    model.eval()
    state, _ = video_env.reset()
    done = False
    while not done:
        with torch.no_grad():
            probs = model(torch.FloatTensor(state).unsqueeze(0))
            action = torch.argmax(probs).item()
        state, _, term, trunc, _ = video_env.step(action)
        done = term or trunc
    
    video_env.close()

    # --- 5. RENDER VIDEO IN JUPYTER ---
    mp4_files = [f for f in os.listdir(video_folder) if f.endswith(".mp4")]
    if mp4_files:
        latest_video = os.path.join(video_folder, sorted(mp4_files)[-1])
        display(Video(latest_video, embed=True, width=600))
    
    return model

# Run it
trained_grpo = train_grpo()

Training GRPO agent...
Iteration 0 | Last Episode Reward: 19.0
Iteration 50 | Last Episode Reward: 13.0
Iteration 100 | Last Episode Reward: 10.0
Iteration 150 | Last Episode Reward: 10.0
Training Complete!

Recording GRPO final result...


> **Category:** On-Policy | Model-Based | Planning
> 
> **The Concept:** **"What if?"** The agent has a mini-simulator in its head. Before it moves, it imagines many futures, picks the best one, and takes the first step.
> 
> **The Steps:**
> 1. **Learn Physics:** The agent learns a **Model** $f_{\phi}(s, a) \approx s'$.
> 2. **Imagine:** At every step, sample $K$ sequences of actions for the next few seconds.
> 3. **Predict:** Use the model to simulate the rewards for each imagined sequence.
> 4. **Select:** Identify the sequence with the highest predicted reward.
> 5. **Execute:** Take only the **first** action $a_t$ of that sequence.
> 6. **Repeat:** Observe the actual $s_{t+1}$ and re-plan everything from scratch.

In [20]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
from IPython.display import Video, display

# --- 1. THE LEARNED MODEL (Environment Model) ---
# This is what the MPC_Agent uses to predict the future: f(s, a) -> s_next
class LearnedDynamics(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(LearnedDynamics, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + 1, 64),
            nn.ReLU(),
            nn.Linear(64, state_dim)
        )
        self.optimizer = optim.Adam(self.parameters(), lr=1e-3)
        self.loss_fn = nn.MSELoss()

    def predict(self, state, action):
        # Helper to make the agent's code work with Torch
        state_t = torch.FloatTensor(state).unsqueeze(0)
        action_t = torch.FloatTensor([[action]])
        with torch.no_grad():
            next_state = self.forward(state_t, action_t)
        
        # Mock reward for CartPole: stay upright (pole angle is index 2)
        # Reward is high if pole angle is close to 0
        reward = 1.0 - abs(next_state[0, 2].item())
        return next_state.numpy().flatten(), reward

    def forward(self, state, action):
        x = torch.cat([state, action], dim=-1)
        return self.net(x)

    def update(self, state, action, next_state):
        state_t = torch.FloatTensor(state).unsqueeze(0)
        action_t = torch.FloatTensor([[action]])
        next_state_t = torch.FloatTensor(next_state).unsqueeze(0)
        
        pred = self.forward(state_t, action_t)
        loss = self.loss_fn(pred, next_state_t)
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

# --- 2. YOUR MPC AGENT ---
class MPC_Agent:
    def __init__(self, env_model):
        self.model = env_model 
    
    def plan(self, current_state, horizon=5):
        best_action = None
        max_reward = -np.inf
        
        # Simple Random Shooting Method
        for _ in range(50): # Reduced to 50 for speed in demo
            sim_state = current_state
            total_reward = 0
            first_action = np.random.choice([0, 1])
            
            for t in range(horizon):
                action = first_action if t == 0 else np.random.choice([0, 1])
                # Agent uses the learned model to "dream"
                sim_state, reward = self.model.predict(sim_state, action)
                total_reward += reward
            
            if total_reward > max_reward:
                max_reward = total_reward
                best_action = first_action
        
        return best_action

# --- 3. TRAINING AND VISUALIZATION ---
def train_and_video_mpc():
    env = gym.make("CartPole-v1")
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    # Initialize the model and the agent
    learned_model = LearnedDynamics(state_dim, action_dim)
    agent = MPC_Agent(learned_model)

    # Phase A: Train the Dynamics Model (Physics)
    print("Learning environment physics...")
    for _ in range(100):
        state, _ = env.reset()
        done = False
        while not done:
            action = env.action_space.sample()
            next_state, _, term, trunc, _ = env.step(action)
            learned_model.update(state, action, next_state)
            state = next_state
            done = term or trunc
    
    print("Training Complete!\n")

    # Phase B: Visualization (MPC Planning)
    print("Recording MPC agent behavior...")
    video_folder = "./mpc_agent_videos"
    video_env = gym.make("CartPole-v1", render_mode="rgb_array")
    video_env = RecordVideo(video_env, video_folder=video_folder, name_prefix="mpc_result", episode_trigger=lambda x: True)

    state, _ = video_env.reset()
    done = False
    while not done:
        # Use your MPC plan logic
        action = agent.plan(state, horizon=10)
        state, _, term, trunc, _ = video_env.step(action)
        done = term or trunc
    
    video_env.close()

    # Phase C: Embed in Jupyter
    mp4_files = [f for f in os.listdir(video_folder) if f.endswith(".mp4")]
    if mp4_files:
        latest_video = os.path.join(video_folder, sorted(mp4_files)[-1])
        display(Video(latest_video, embed=True, width=600))

# Run the process
train_and_video_mpc()

Learning environment physics...
Training Complete!

Recording MPC agent behavior...


> **Category:** On-Policy | Reference-Anchored | Alignment
> 
> **The Concept:** **"Stay True to Your Roots."** Instead of relying on a separate "Critic" to tell it how good a state is, the agent compares its current choices against a version of its former self (the Reference Model). It updates by leaning toward actions that yield higher rewards than the reference while ensuring it doesn't "forget" its fundamental behavior or become unstable.
> 
> **The Steps:**
> 1. **The Twin:** The agent maintains a **Reference Model** $\pi_{\text{ref}}$, which is a frozen snapshot of the policy before the current training phase.
> 2. **Act & Observe:** The agent takes actions in the environment to collect rewards, just like standard RL.
> 3. **The Comparison:** For every action taken, the agent calculates the **Log-Ratio**: how much more (or less) likely it is to take that action now compared to the Reference Model.
> 4. **Preference Weighting:** Rewards are treated as "preference signals." If an action yields a high reward, the agent pushes its current policy to increase the log-ratio (moving further from the reference in a positive direction).
> 5. **Dampened Optimization:** Using a $\beta$ (KL-penalty) coefficient, the agent is penalized if it drifts too far from the reference, preventing the "policy collapse" common in noisy environments.
> 6. **Update the Anchor:** Periodically, the Reference Model is updated to the current Policy's weights to allow for continuous improvement.

In [ ]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import os
import copy
from IPython.display import Video, display

# --- 1. THE POLICY NETWORK ---
class DAPO_Policy(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(DAPO_Policy, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        return self.net(x)

# --- 2. THE DAPO UPDATE LOGIC ---
def dapo_update(model, ref_model, optimizer, states, actions, rewards, beta=0.1):
    """
    DAPO utilizes a reference model (ref_model) to anchor the update.
    It optimizes the log-ratio of the current policy vs the reference policy
    weighted by the reward/preference.
    """
    # Current policy log probabilities
    probs = model(states)
    dist = torch.distributions.Categorical(probs)
    log_probs = dist.log_prob(actions)

    # Reference policy log probabilities (Frozen)
    with torch.no_grad():
        ref_probs = ref_model(states)
        ref_dist = torch.distributions.Categorical(ref_probs)
        ref_log_probs = ref_dist.log_prob(actions)

    # DAPO Loss: Similar to DPO but applied to environment rewards
    # We maximize: Reward * log(pi/pi_ref) - penalty
    # Here simplified as a policy gradient weighted by the relative log-ratio
    ratio = log_probs - ref_log_probs
    
    # We treat high reward as 'Preferred' and low reward as 'Rejected'
    # For CartPole, we use the advantage (reward - baseline) to scale the alignment
    reward_tensor = torch.FloatTensor(rewards)
    baseline = reward_tensor.mean()
    advantage = reward_tensor - baseline

    # The DAPO objective minimizes the divergence while maximizing reward
    loss = -(advantage * torch.sigmoid(beta * ratio)).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# --- 3. TRAINING LOOP ---
def train_dapo():
    env = gym.make("CartPole-v1")
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    model = DAPO_Policy(state_dim, action_dim)
    # The Reference model starts as a copy of the initial model
    ref_model = copy.deepcopy(model)
    ref_model.eval() 
    
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    batch_size = 16 
    print("Training DAPO agent...")
    
    for iteration in range(200):
        state, _ = env.reset()
        done = False
        total_iter_reward = 0
        
        while not done:
            # Batching states for a single step update (simulating multiple trajectories)
            state_t = torch.FloatTensor(state).repeat(batch_size, 1)
            
            with torch.no_grad():
                probs = model(state_t)
                dist = torch.distributions.Categorical(probs)
                actions = dist.sample()
            
            # Environment step
            next_state, reward, term, trunc, _ = env.step(actions[0].item())
            done = term or trunc
            
            # Mocking preferences: Actions that result in staying upright 
            # are compared against a synthetic "worse" batch
            rewards = [reward if i % 2 == 0 else (reward * 0.5) for i in range(batch_size)]
            
            # Update DAPO
            dapo_update(model, ref_model, optimizer, state_t, actions, rewards)
            
            state = next_state
            total_iter_reward += reward

        # Periodically update the reference model to the current policy (SFT-style)
        if iteration % 50 == 0:
            ref_model.load_state_dict(model.state_dict())
            print(f"Iteration {iteration} | Last Episode Reward: {total_iter_reward}")

    env.close()
    print("Training Complete!\n")

    # --- 4. VISUALIZATION ---
    video_folder = "./dapo_videos"
    video_env = gym.make("CartPole-v1", render_mode="rgb_array")
    video_env = RecordVideo(video_env, video_folder=video_folder, name_prefix="dapo_result", episode_trigger=lambda x: True)

    model.eval()
    state, _ = video_env.reset()
    done = False
    while not done:
        with torch.no_grad():
            probs = model(torch.FloatTensor(state).unsqueeze(0))
            action = torch.argmax(probs).item()
        state, _, term, trunc, _ = video_env.step(action)
        done = term or trunc
    
    video_env.close()

    mp4_files = [f for f in os.listdir(video_folder) if f.endswith(".mp4")]
    if mp4_files:
        latest_video = os.path.join(video_folder, sorted(mp4_files)[-1])
        display(Video(latest_video, embed=True, width=600))
    
    return model

# Run it
trained_dapo = train_dapo()

> **Category:** On-Policy | Efficiency-Focused | Localized RL
> 
> **The Concept:** **"Focus on the Hard Parts."** Instead of wasting compute on easy steps the agent already knows or impossible ones it can't solve, PivotRL identifies "Pivots"—specific moments in a task where the agent's choices actually make a difference. It uses existing expert data to jump straight to these critical moments and practices there.
> 
> **The Steps:**
> 1. **Extract Candidates:** Take successful expert trajectories (from SFT data) and break them into individual turns $(s_t, a_t)$.
> 2. **Profile & Filter:** Run a quick test on each turn. If the agent always succeeds or always fails, discard it. Keep only the **Pivots**: turns where the agent shows high variance (some attempts work, others don't).
> 3. **Local Rollout:** Instead of starting from the beginning, teleport the agent directly to a Pivot state $s_t$ and have it sample multiple completions.
> 4. **Functional Scoring:** Don't require the agent to mimic the expert's exact words. Use a verifier to reward any action that is "functionally equivalent" (e.g., a different code command that gets the same result).
> 5. **Group Update:** Use GRPO to update the policy based on the relative success of these local attempts, maximizing the learning signal from that specific decision point.
> 6. **Efficiency Gain:** By practicing only at these "Pivot" points, achieve the accuracy of end-to-end RL with 4x less computation.

In [21]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
from IPython.display import Video, display

# --- 1. THE POLICY NETWORK ---
class PivotRL_Policy(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(PivotRL_Policy, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        return self.net(x)

# --- 2. PIVOT-BASED GROUP UPDATE (GRPO Style) ---
def pivot_update(model, optimizer, states, actions, old_log_probs, rewards_group):
    # Relative advantage within the local pivot group
    mean_reward = np.mean(rewards_group)
    std_reward = np.std(rewards_group) + 1e-8
    advantages = torch.FloatTensor([(r - mean_reward) / std_reward for r in rewards_group])

    probs = model(states)
    dist = torch.distributions.Categorical(probs)
    new_log_probs = dist.log_prob(actions)

    # Standard Policy Gradient update for the local pivot
    ratio = torch.exp(new_log_probs - old_log_probs)
    loss = -(ratio * advantages).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# --- 3. TRAINING LOOP ---
def train_pivot_rl():
    env = gym.make("CartPole-v1")
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    model = PivotRL_Policy(state_dim, action_dim)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    # "Expert" Pivots: Pre-saved states where the pole is leaning (critical moments)
    # In a real LLM, these would be turns extracted from SFT data.
    pivots = [
        np.array([0.01, 0.02, 0.05, 0.02]),   # Slightly leaning right
        np.array([-0.01, -0.02, -0.05, -0.02]) # Slightly leaning left
    ]
    
    group_size = 8
    print("Training PivotRL agent...")

    for iteration in range(200):
        # 1. Pick a 'Pivot' state instead of env.reset() every time
        pivot_state = pivots[np.random.choice(len(pivots))]
        
        # 2. Local Rollout: Sample multiple completions from this pivot
        state_t = torch.FloatTensor(pivot_state).repeat(group_size, 1)
        
        with torch.no_grad():
            probs = model(state_t)
            dist = torch.distributions.Categorical(probs)
            actions = dist.sample()
            old_log_probs = dist.log_prob(actions)
            
        # 3. Functional Scoring: Execute actions and get local rewards
        rewards = []
        for i in range(group_size):
            # Reset env to pivot state manually (simplified for demo)
            env.reset()
            env.unwrapped.state = pivot_state 
            _, reward, term, trunc, _ = env.step(actions[i].item())
            rewards.append(reward)
            
        # 4. Localized Update
        pivot_update(model, optimizer, state_t, actions, old_log_probs, rewards)

        if iteration % 50 == 0:
            print(f"Iteration {iteration} | Local Pivot Reward Mean: {np.mean(rewards)}")

    env.close()
    print("Training Complete!\n")

    # --- 4. VISUALIZATION ---
    print("Recording PivotRL final result...")
    video_folder = "./pivot_videos"
    video_env = gym.make("CartPole-v1", render_mode="rgb_array")
    video_env = RecordVideo(video_env, video_folder=video_folder, name_prefix="pivot_result", episode_trigger=lambda x: True)

    model.eval()
    state, _ = video_env.reset()
    done = False
    while not done:
        with torch.no_grad():
            probs = model(torch.FloatTensor(state).unsqueeze(0))
            action = torch.argmax(probs).item()
        state, _, term, trunc, _ = video_env.step(action)
        done = term or trunc
    
    video_env.close()

    mp4_files = [f for f in os.listdir(video_folder) if f.endswith(".mp4")]
    if mp4_files:
        latest_video = os.path.join(video_folder, sorted(mp4_files)[-1])
        display(Video(latest_video, embed=True, width=600))
    
    return model

# Run it
trained_pivot = train_pivot_rl()

Training PivotRL agent...
Iteration 0 | Local Pivot Reward Mean: 1.0
Iteration 50 | Local Pivot Reward Mean: 1.0
Iteration 100 | Local Pivot Reward Mean: 1.0
Iteration 150 | Local Pivot Reward Mean: 1.0
Training Complete!

Recording PivotRL final result...
